<a href="https://colab.research.google.com/github/Innovatewithapple/Resume_JD_AnalysisTransfomer/blob/main/Resume_JD_Encoder_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import random
import numpy as np
import os

def setupSeed(seed=32):
  os.environ['PYTHONHASHSEED'] = str(seed)
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True

setupSeed()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from datasets import load_dataset
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from transformers import AutoTokenizer,AutoModel
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
import gc
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
ds = load_dataset("cnamuangtoun/resume-job-description-fit")

train.csv:   0%|          | 0.00/53.4M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/15.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6241 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1759 [00:00<?, ? examples/s]

In [ ]:
resume_texts = ds['train']['resume_text']
jd_texts = ds['train']['job_description_text']
label_text = ds['train']['label']

In [ ]:
val_resume_texts = ds['test']['resume_text']
val_jd_texts = ds['test']['job_description_text']
val_label_text = ds['test']['label']

In [ ]:
#-------Auto Tokenizer--------!
autoToken = AutoTokenizer.from_pretrained('bert-base-uncased')

#--------Model-------------!
encoder_model = AutoModel.from_pretrained('bert-base-uncased')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
labelencoder = LabelEncoder()
label_encoded_text = labelencoder.fit_transform(label_text)
val_label_encoded_text = labelencoder.transform(val_label_text)

In [ ]:
train_encoding = autoToken(text=list(resume_texts),text_pair=list(jd_texts),padding=True,max_length=512,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')
val_encoding = autoToken(text=list(val_resume_texts),text_pair=list(val_jd_texts),padding=True,max_length=512,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

In [ ]:
class TokenAttentionLayer(Dataset):
  def __init__(self,encoding,label):
    self.encoding = encoding
    self.label = label

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    input_ids = self.encoding['input_ids'][idx]
    attention_mask = self.encoding['attention_mask'][idx]
    labels = torch.tensor(self.label[idx],dtype=torch.long)

    return{
        'input_ids':input_ids,
        'attention_mask':attention_mask,
        'labels':labels
    }

In [ ]:
train_dataset = TokenAttentionLayer(train_encoding,label_encoded_text)
val_dataset = TokenAttentionLayer(val_encoding,val_label_encoded_text)

In [ ]:
train_loader = DataLoader(dataset=train_dataset,batch_size=32,shuffle=True,pin_memory=True,num_workers=0)
val_loader = DataLoader(dataset=val_dataset,batch_size=32,shuffle=False,pin_memory=True,num_workers=0)

In [ ]:
class DeBertPretrained(nn.Module):
  def __init__(self,encoder_model,num_classes):
    super().__init__()
    self.encoder = encoder_model
    self.dropout = nn.Dropout(0.2)
    self.classifier = nn.Linear(encoder_model.config.hidden_size,num_classes)

  def forward(self,input_ids,attention_mask):
    x = self.encoder(input_ids=input_ids,attention_mask=attention_mask)
    x = x.last_hidden_state

    mean = attention_mask.unsqueeze(-1)
    output = (x*mean).sum(dim=1) / mean.sum(dim=1)

    return self.classifier(self.dropout(output))

In [ ]:
model = DeBertPretrained(encoder_model=encoder_model,num_classes=3)

for parameter in model.encoder.parameters():
  parameter.required_grad = False

for param in model.encoder.encoder.layer[-2:].parameters():
    param.requires_grad = True

In [ ]:
classes = np.unique(label_encoded_text)
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=label_encoded_text
)

print(weights)

[1.34911371 0.66189416 1.33697515]


In [ ]:
# MULTI GPU SUPPORT
# -----------------------------------
# if torch.cuda.device_count() > 1:
#     print("Using Multiple GPUs")
#     model = nn.DataParallel(model)

# Move FINAL model to device
model = model.to(device)
optimizer = optim.AdamW(params=filter(lambda p: p.requires_grad, model.parameters()),lr=1e-3,weight_decay=1e-2)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)
# loss_fn = nn.CrossEntropyLoss()
scalar = torch.amp.GradScaler('cuda')

In [ ]:
epochs = 5

for epoch in range(epochs):
  model.train()
  train_loss = 0
  train_Correct = 0
  train_total = 0
  progress_bar_train = tqdm(train_loader, desc=f"Epoch {epoch+1}")

  for batch in progress_bar_train:
    text = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    optimizer.zero_grad()
    with torch.amp.autocast('cuda'):
      output = model(input_ids=text,attention_mask=attention_mask)
      loss = loss_fn(output,labels)

    scalar.scale(loss).backward()
    scalar.step(optimizer)
    scalar.update()

    train_loss +=loss.item()
    pred = torch.argmax(output,1)
    train_Correct += (pred == labels).sum().item()
    train_total += labels.size(0)

  train_accuracy = train_Correct / train_total
  train_losses = train_loss / len(train_loader)

  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0
  progress_bar_val = tqdm(val_loader, desc='Validation')
  with torch.no_grad():
    for batch in progress_bar_val:
      text = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device)
      labels = batch['labels'].to(device)

      output = model(input_ids=text,attention_mask=attention_mask)
      loss = loss_fn(output,labels)
      val_loss += loss.item()

      pred = torch.argmax(output,1)
      val_correct += (pred == labels).sum().item()
      val_total += labels.size(0)

  val_accuracy = val_correct / val_total
  val_losses = val_loss / len(val_loader)

  print(f'\nEpochs: {epoch+1}/{epochs}...')
  print(f'Train_acc: {train_accuracy} | Train_loss: {train_losses}')
  print(f'Val_acc: {val_accuracy} | Val_loss: {val_losses}')
  print('==============================================================')


Validation: 100%|██████████| 55/55 [00:54<00:00,  1.01it/s]



Epochs: 1/5...
Train_acc: 0.3626021470918122 | Train_loss: 1.1028590652407433
Val_acc: 0.4872086412734508 | Val_loss: 1.0922993226484818


Validation: 100%|██████████| 55/55 [00:54<00:00,  1.01it/s]



Epochs: 2/5...
Train_acc: 0.32542861720878064 | Train_loss: 1.1040118458319683
Val_acc: 0.4872086412734508 | Val_loss: 1.09755248373205


Validation: 100%|██████████| 55/55 [00:54<00:00,  1.01it/s]



Epochs: 3/5...
Train_acc: 0.34898253485018427 | Train_loss: 1.1020906871678877
Val_acc: 0.4872086412734508 | Val_loss: 1.0922168319875545


Epoch 4:  76%|███████▌  | 149/196 [01:49<00:34,  1.36it/s]


KeyboardInterrupt: 

In [ ]:
  gc.collect()
  torch.cuda.empty_cache()

In [ ]:
# Check actual distribution
import pandas as pd
df_train = pd.DataFrame(ds['train'])
print(df_train['label'].value_counts())
print(df_train['label'].value_counts(normalize=True))

label
No Fit           3143
Potential Fit    1556
Good Fit         1542
Name: count, dtype: int64
label
No Fit           0.503605
Potential Fit    0.249319
Good Fit         0.247076
Name: proportion, dtype: float64
